# Getting Started with Pose-Independent 3D Anthropometry

This notebook provides a comprehensive introduction to the **Pose-Independent 3D Anthropometry** repository, which implements the paper ["Pose-independent 3D Anthropometry from Sparse Data"](https://inria.hal.science/hal-04683475/file/eccv_2024_pose_independent_anthropometry_camera_ready.pdf) presented at ECCV 2024.

## 🎯 Project Overview

**Goal**: Estimate 11 body measurements from 70 body landmarks of a posed subject.

**Key Features**:
- Works with posed subjects (not just A-pose)
- Uses sparse landmark data (70 landmarks)
- Estimates anthropometric measurements in a pose-independent manner
- Supports multiple datasets: CAESAR, DYNA, and 4DHumanOutfit

**Measurements Estimated**:
1. Ankle Circumference
2. Arm Length (Shoulder to Elbow)
3. Arm Length (Shoulder to Wrist)
4. Arm Length (Spine to Wrist)
5. Chest Circumference
6. Crotch Height
7. Head Circumference
8. Hip Circ Max Height
9. Hip Circumference, Maximum
10. Neck Base Circumference
11. Stature (Height)

## 🛠️ Environment Setup

First, let's install the required dependencies and set up our environment.

In [ ]:
# Install required packages
import subprocess
import sys

def install_package(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])

# Install key dependencies
packages = [
    "torch",
    "numpy",
    "pandas",
    "matplotlib",
    "seaborn",
    "pyyaml",
    "tqdm",
    "scipy",
    "scikit-learn"
]

for package in packages:
    try:
        __import__(package.replace("-", "_"))
        print(f"✓ {package} already installed")
    except ImportError:
        print(f"Installing {package}...")
        install_package(package)

In [ ]:
# Import necessary libraries
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import yaml
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set up plotting
plt.style.use('default')
sns.set_palette("husl")
%matplotlib inline

print("Environment setup complete!")
print(f"PyTorch version: {torch.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Current working directory: {os.getcwd()}")

## 📁 Repository Structure

Let's explore the repository structure to understand the codebase.

In [ ]:
# Display repository structure
def show_tree(path, prefix="", max_depth=2, current_depth=0):
    if current_depth >= max_depth:
        return
    
    items = sorted(Path(path).iterdir())
    dirs = [item for item in items if item.is_dir() and not item.name.startswith('.')]
    files = [item for item in items if item.is_file() and not item.name.startswith('.')]
    
    # Show directories first
    for i, item in enumerate(dirs):
        is_last = i == len(dirs) - 1 and len(files) == 0
        print(f"{prefix}{'└── ' if is_last else '├── '}{item.name}/")
        extension = "    " if is_last else "│   "
        show_tree(item, prefix + extension, max_depth, current_depth + 1)
    
    # Show files
    for i, item in enumerate(files):
        is_last = i == len(files) - 1
        print(f"{prefix}{'└── ' if is_last else '├── '}{item.name}")

print("📁 Repository Structure:")
print("pose-independent-anthropometry/")
show_tree(".", max_depth=2)

## 🔧 Configuration Overview

Let's examine the configuration file to understand the model parameters and settings.

In [ ]:
# Load and display configuration
with open('configs/config_real.yaml', 'r') as f:
    config = yaml.safe_load(f)

print("🔧 Configuration Overview:")
print("=" * 50)

# Display key configuration sections
sections_to_show = ['general', 'learning', 'model_configs']

for section in sections_to_show:
    if section in config:
        print(f"\n📋 {section.upper()}:")
        for key, value in config[section].items():
            if isinstance(value, list) and len(value) > 3:
                print(f"  {key}: [{value[0]}, {value[1]}, ..., {value[-1]}] ({len(value)} items)")
            else:
                print(f"  {key}: {value}")

# Show measurements
print(f"\n📏 MEASUREMENTS ({len(config['learning']['measurements'])}):") 
for i, measurement in enumerate(config['learning']['measurements'], 1):
    print(f"  {i:2d}. {measurement}")

## 🧠 Model Architecture

Let's examine and visualize the SimpleMLP model architecture used for anthropometric estimation.

In [ ]:
# Import the model
from models import SimpleMLP

# Create model instance with default parameters
model_config = config['model_configs']['SimpleMLP']
model = SimpleMLP(
    encoder_input_dim=368,  # Default input dimension
    output_dim=model_config['output_dim'],
    hidden_dim1=model_config['hidden_dim1'],
    hidden_dim2=model_config['hidden_dim2']
)

print("🧠 SimpleMLP Model Architecture:")
print("=" * 40)
print(model)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n📊 Model Statistics:")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")

# Visualize architecture
fig, ax = plt.subplots(1, 1, figsize=(12, 6))

# Define layer information
layers = [
    ("Input\n(Landmarks)", 368, "lightblue"),
    ("Hidden 1\n(ReLU)", model_config['hidden_dim1'], "lightgreen"),
    ("Hidden 2\n(ReLU)", model_config['hidden_dim2'], "lightcoral"),
    ("Output\n(Measurements)", model_config['output_dim'], "lightyellow")
]

# Plot architecture
x_positions = np.arange(len(layers))
y_positions = [layer[1] for layer in layers]
colors = [layer[2] for layer in layers]
labels = [layer[0] for layer in layers]

bars = ax.bar(x_positions, y_positions, color=colors, alpha=0.7, edgecolor='black')

# Add value labels on bars
for i, (bar, value) in enumerate(zip(bars, y_positions)):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, 
            str(value), ha='center', va='bottom', fontweight='bold')

ax.set_xticks(x_positions)
ax.set_xticklabels(labels)
ax.set_ylabel('Number of Neurons')
ax.set_title('SimpleMLP Architecture for Pose-Independent Anthropometry', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 📊 Understanding the Data Pipeline

Let's explore how the data flows through the system and what the input features look like.

In [ ]:
# Import utilities to understand landmarks and measurements
try:
    from utils import SMPL_INDEX_LANDAMRKS_REVISED
    
    print("🎯 SMPL Landmarks Used:")
    print("=" * 30)
    
    landmark_names = list(SMPL_INDEX_LANDAMRKS_REVISED.keys())
    print(f"Total landmarks: {len(landmark_names)}")
    
    # Display landmarks in a nice format
    for i, landmark in enumerate(landmark_names, 1):
        if i % 3 == 1:
            print(f"\n{i:2d}. {landmark:<25}", end="")
        else:
            print(f"{i:2d}. {landmark:<25}", end="")
    
    print("\n")
    
except ImportError:
    print("⚠️ Could not import landmark definitions. This is normal if SMPL models are not set up.")
    print("The system uses 70 anatomical landmarks on the human body.")

In [ ]:
# Demonstrate feature transformation pipeline
print("🔄 Feature Transformation Pipeline:")
print("=" * 40)

# Simulate landmark data (70 landmarks × 3 coordinates = 210 features)
n_landmarks = 70
n_coords = 3
simulated_landmarks = np.random.randn(n_landmarks, n_coords) * 100  # Simulate in mm

print(f"1. Raw Landmarks: {simulated_landmarks.shape} -> {n_landmarks} landmarks × {n_coords} coordinates")

# Coordinate features (flattened)
coord_features = simulated_landmarks.flatten()
print(f"2. Coordinate Features: {coord_features.shape} -> Flattened coordinates")

# Distance features (pairwise distances)
from scipy.spatial.distance import pdist
distance_features = pdist(simulated_landmarks)
print(f"3. Distance Features: {distance_features.shape} -> Pairwise distances between landmarks")

# Combined features (as used by the model)
total_features = len(coord_features) + len(distance_features)
print(f"4. Combined Features: {total_features} -> Input to SimpleMLP")

print(f"\n📈 Feature Breakdown:")
print(f"  • Coordinate features: {len(coord_features)} ({len(coord_features)/total_features*100:.1f}%)")
print(f"  • Distance features: {len(distance_features)} ({len(distance_features)/total_features*100:.1f}%)")
print(f"  • Total input features: {total_features}")

## 🎯 Using the Pre-trained Model

Let's load and use the pre-trained model to make predictions.

In [ ]:
# Check for pre-trained model
pretrained_path = "results/2024_07_11_09_42_48"

if os.path.exists(pretrained_path):
    print(f"✓ Found pre-trained model at: {pretrained_path}")
    
    # List contents
    model_files = list(Path(pretrained_path).glob("*.pth"))
    if model_files:
        print(f"  Model files found: {[f.name for f in model_files]}")
        
        # Try to load the model
        try:
            model_file = model_files[0]  # Use first model file
            checkpoint = torch.load(model_file, map_location='cpu')
            
            # Create model and load weights
            model = SimpleMLP(
                encoder_input_dim=368,
                output_dim=11,
                hidden_dim1=194,
                hidden_dim2=97
            )
            
            if 'model_state_dict' in checkpoint:
                model.load_state_dict(checkpoint['model_state_dict'])
            else:
                model.load_state_dict(checkpoint)
            
            model.eval()
            print(f"✓ Successfully loaded pre-trained model from {model_file.name}")
            
            # Test with dummy data
            dummy_input = torch.randn(1, 368)  # Batch size 1, 368 features
            with torch.no_grad():
                predictions = model(dummy_input)
            
            print(f"\n🧪 Test Prediction (dummy data):")
            measurements = config['learning']['measurements']
            for i, (measurement, value) in enumerate(zip(measurements, predictions[0])):
                print(f"  {i+1:2d}. {measurement}: {value.item():.2f} mm")
                
        except Exception as e:
            print(f"⚠️ Could not load model: {e}")
            print("This is normal if the model format is different or dependencies are missing.")
    else:
        print("  No .pth model files found in the directory")
else:
    print(f"⚠️ Pre-trained model not found at: {pretrained_path}")
    print("You can train a new model using the training section below.")

## 📚 Dataset Information

Let's explore the datasets supported by this repository.

In [ ]:
# Dataset information
datasets_info = {
    "CAESAR": {
        "description": "Civilian American and European Surface Anthropometry Resource",
        "type": "Static 3D body scans",
        "subjects": "~4,400 subjects",
        "availability": "Commercial",
        "use_case": "Training and evaluation"
    },
    "DYNA": {
        "description": "Dynamic human body in motion",
        "type": "4D sequences (3D + time)",
        "subjects": "10 subjects",
        "availability": "Free (registration required)",
        "use_case": "Evaluation on dynamic sequences"
    },
    "4DHumanOutfit": {
        "description": "4D humans in clothing performing various actions",
        "type": "4D clothed human sequences",
        "subjects": "6 subjects",
        "availability": "Free (upon request)",
        "use_case": "Evaluation on clothed subjects"
    }
}

print("📚 Supported Datasets:")
print("=" * 50)

for dataset_name, info in datasets_info.items():
    print(f"\n🗂️ {dataset_name}:")
    for key, value in info.items():
        print(f"  {key.capitalize()}: {value}")

# Create a visualization of dataset characteristics
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Dataset sizes
dataset_names = list(datasets_info.keys())
subject_counts = [4400, 10, 6]  # Approximate subject counts

bars1 = ax1.bar(dataset_names, subject_counts, color=['skyblue', 'lightgreen', 'lightcoral'])
ax1.set_ylabel('Number of Subjects')
ax1.set_title('Dataset Sizes', fontweight='bold')
ax1.set_yscale('log')

# Add value labels
for bar, count in zip(bars1, subject_counts):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.1, 
             str(count), ha='center', va='bottom', fontweight='bold')

# Dataset types
dataset_types = ['Static 3D', '4D Dynamic', '4D Clothed']
type_colors = ['skyblue', 'lightgreen', 'lightcoral']

ax2.pie([1, 1, 1], labels=dataset_types, colors=type_colors, autopct='',
        startangle=90, textprops={'fontsize': 10})
ax2.set_title('Dataset Types', fontweight='bold')

plt.tight_layout()
plt.show()

## 🏋️ Training Process Overview

Let's understand how to train the model and what the training process involves.

In [ ]:
# Training process overview
print("🏋️ Training Process Overview:")
print("=" * 40)

training_steps = [
    "1. Data Preparation",
    "   • Load CAESAR dataset with landmarks and measurements",
    "   • Create posed versions using SMPL body model",
    "   • Extract coordinate and distance features",
    "   • Normalize landmarks to pelvis",
    "",
    "2. Model Configuration",
    "   • SimpleMLP with 368 input features",
    "   • Hidden layers: 194 → 97 neurons",
    "   • Output: 11 anthropometric measurements",
    "   • ReLU activations",
    "",
    "3. Training Setup",
    f"   • Batch size: {config['learning']['batch_size']:,}",
    f"   • Learning rate: {config['learning']['init_lr']}",
    f"   • Epochs: {config['learning']['nepoch']:,}",
    "   • Loss: Mean Squared Error (MSE)",
    "",
    "4. Evaluation",
    "   • Test on multiple datasets and poses",
    "   • Metrics: MAE, RMSE, relative error",
    "   • Compare with baseline methods"
]

for step in training_steps:
    print(step)

# Show training command
print("\n💻 Training Command:")
print("```bash")
print("python train.py")
print("```")

print("\n📊 Monitoring Training:")
print("```bash")
print("# In a separate terminal:")
print("visdom -p 8080")
print("# Then open: http://localhost:8080")
print("```")

## 📈 Evaluation and Results

Let's understand how to evaluate the model and interpret results.

In [ ]:
# Evaluation scenarios from the paper
evaluation_scenarios = {
    "CAESAR A-pose": {
        "description": "Standard A-pose evaluation",
        "command": "python evaluate.py CAESAR_STAND -R results/2024_07_11_09_42_48",
        "table": "Tables 1 & 2 (left)"
    },
    "CAESAR A-pose + Noise": {
        "description": "A-pose with 5mm landmark noise",
        "command": "python evaluate.py CAESAR_NOISY -R results/2024_07_11_09_42_48 --pelvis_normalization",
        "table": "Tables 1 & 2 (right)"
    },
    "CAESAR Sitting": {
        "description": "Sitting B-pose evaluation",
        "command": "python evaluate.py CAESAR_SIT_TRANS_BM -R results/2024_07_11_09_42_48",
        "table": "Table 3"
    },
    "CAESAR Arbitrary Pose": {
        "description": "Various arbitrary poses",
        "command": "python evaluate.py CAESAR_POSED -R results/2024_07_11_09_42_48",
        "table": "Table 4"
    },
    "DYNA Dynamic": {
        "description": "Dynamic motion sequences",
        "command": "python evaluate.py DYNA -R results/2024_07_11_09_42_48",
        "table": "Table 5"
    },
    "4DHumanOutfit": {
        "description": "Clothed subjects in motion",
        "command": "python evaluate.py FOUR_D_HUMAN_OUTFIT -R results/2024_07_11_09_42_48",
        "table": "Table 6"
    }
}

print("📈 Evaluation Scenarios:")
print("=" * 50)

for scenario, info in evaluation_scenarios.items():
    print(f"\n🎯 {scenario}:")
    print(f"  Description: {info['description']}")
    print(f"  Paper reference: {info['table']}")
    print(f"  Command: {info['command']}")

# Create a visualization of evaluation complexity
fig, ax = plt.subplots(1, 1, figsize=(12, 8))

scenarios = list(evaluation_scenarios.keys())
complexity_scores = [1, 2, 3, 4, 5, 6]  # Relative complexity
colors = plt.cm.viridis(np.linspace(0, 1, len(scenarios)))

bars = ax.barh(scenarios, complexity_scores, color=colors, alpha=0.7)

ax.set_xlabel('Evaluation Complexity')
ax.set_title('Evaluation Scenarios by Complexity', fontweight='bold', fontsize=14)
ax.set_xlim(0, 7)

# Add complexity labels
complexity_labels = ['Basic', 'Noise Robust', 'Pose Variant', 'Multi-pose', 'Dynamic', 'Clothed']
for bar, label in zip(bars, complexity_labels):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2, 
            label, va='center', fontsize=10)

plt.tight_layout()
plt.show()

## 🔬 Example: Simulated Inference

Let's create a simple example of how the model would work with simulated data.

In [ ]:
# Create a simple inference example with simulated data
def simulate_anthropometry_prediction():
    """Simulate the anthropometry prediction process"""
    
    # Create model
    model = SimpleMLP(encoder_input_dim=368, output_dim=11, hidden_dim1=194, hidden_dim2=97)
    model.eval()
    
    # Simulate different body types
    body_types = {
        "Average Adult": torch.randn(1, 368) * 0.5,
        "Tall Person": torch.randn(1, 368) * 0.5 + 0.3,
        "Short Person": torch.randn(1, 368) * 0.5 - 0.3,
        "Athletic Build": torch.randn(1, 368) * 0.3 + 0.1
    }
    
    measurements = config['learning']['measurements']
    
    print("🔬 Simulated Anthropometry Predictions:")
    print("=" * 60)
    
    results = {}
    
    for body_type, features in body_types.items():
        with torch.no_grad():
            predictions = model(features)
        
        # Convert to reasonable measurement values (simulate realistic scaling)
        scaled_predictions = torch.abs(predictions) * 50 + torch.tensor([
            250,  # Ankle Circumference
            330,  # Arm Length (Shoulder to Elbow)
            610,  # Arm Length (Shoulder to Wrist)
            810,  # Arm Length (Spine to Wrist)
            980,  # Chest Circumference
            780,  # Crotch Height
            560,  # Head Circumference
            840,  # Hip Circ Max Height
            1020, # Hip Circumference, Maximum
            440,  # Neck Base Circumference
            1700  # Stature
        ])
        
        results[body_type] = scaled_predictions[0]
        
        print(f"\n👤 {body_type}:")
        for i, (measurement, value) in enumerate(zip(measurements, scaled_predictions[0])):
            unit = "mm"
            if "Stature" in measurement:
                print(f"  {i+1:2d}. {measurement}: {value.item():.0f} {unit} ({value.item()/10:.1f} cm)")
            else:
                print(f"  {i+1:2d}. {measurement}: {value.item():.0f} {unit}")
    
    return results

# Run simulation
simulation_results = simulate_anthropometry_prediction()

In [ ]:
# Visualize the simulated results
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
axes = axes.flatten()

measurements = config['learning']['measurements']
body_types = list(simulation_results.keys())
colors = ['skyblue', 'lightgreen', 'lightcoral', 'lightyellow']

# Plot each body type
for idx, (body_type, color) in enumerate(zip(body_types, colors)):
    values = simulation_results[body_type].numpy()
    
    # Create abbreviated measurement names for plotting
    short_names = [name.split('(')[0].strip()[:15] + '...' if len(name) > 15 
                   else name.split('(')[0].strip() for name in measurements]
    
    bars = axes[idx].bar(range(len(values)), values, color=color, alpha=0.7, edgecolor='black')
    axes[idx].set_title(f'{body_type}', fontweight='bold')
    axes[idx].set_ylabel('Measurement (mm)')
    axes[idx].set_xticks(range(len(short_names)))
    axes[idx].set_xticklabels(short_names, rotation=45, ha='right', fontsize=8)
    axes[idx].grid(axis='y', alpha=0.3)
    
    # Add value labels on bars for key measurements
    key_indices = [10, 4, 8]  # Stature, Chest, Hip circumference
    for i in key_indices:
        axes[idx].text(bars[i].get_x() + bars[i].get_width()/2, 
                      bars[i].get_height() + 20, 
                      f'{values[i]:.0f}', ha='center', va='bottom', 
                      fontsize=8, fontweight='bold')

plt.suptitle('Simulated Anthropometric Measurements by Body Type', 
             fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

# Create a comparison plot
fig, ax = plt.subplots(1, 1, figsize=(14, 8))

x = np.arange(len(measurements))
width = 0.2

for i, (body_type, color) in enumerate(zip(body_types, colors)):
    values = simulation_results[body_type].numpy()
    ax.bar(x + i*width, values, width, label=body_type, color=color, alpha=0.7)

ax.set_xlabel('Measurements')
ax.set_ylabel('Value (mm)')
ax.set_title('Comparison of Anthropometric Measurements Across Body Types', fontweight='bold')
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels([name.split('(')[0].strip()[:12] + '...' if len(name) > 12 
                    else name.split('(')[0].strip() for name in measurements], 
                   rotation=45, ha='right')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 🚀 Next Steps and Advanced Usage

Here are the next steps you can take to work with this repository:

In [ ]:
# Next steps guide
next_steps = {
    "🏗️ Setup for Real Data": [
        "1. Download SMPL body models from https://smpl.is.tue.mpg.de/",
        "2. Place SMPL models in data/body_models/smpl/",
        "3. Download pose data and prior from provided links",
        "4. Initialize git submodules: git submodule update --init --recursive"
    ],
    
    "📊 Working with Datasets": [
        "1. Obtain CAESAR dataset (commercial) or DYNA dataset (free)",
        "2. Use SMPL-Fitting repository to fit body models to scans",
        "3. Run dataset creation scripts in scripts/ folder",
        "4. Update paths in configs/config_real.yaml"
    ],
    
    "🏋️ Training Your Own Model": [
        "1. Prepare training data using provided scripts",
        "2. Adjust configuration in configs/config_real.yaml",
        "3. Run: python train.py",
        "4. Monitor training with visdom: visdom -p 8080"
    ],
    
    "🔬 Custom Evaluation": [
        "1. Create your own test datasets",
        "2. Modify evaluate.py for custom metrics",
        "3. Run evaluation on different pose conditions",
        "4. Compare with baseline methods"
    ],
    
    "🛠️ Extending the Method": [
        "1. Experiment with different model architectures in models.py",
        "2. Add new feature transformations in utils.py",
        "3. Implement additional loss functions in losses.py",
        "4. Add support for new datasets in dataset.py"
    ]
}

print("🚀 Next Steps and Advanced Usage:")
print("=" * 50)

for category, steps in next_steps.items():
    print(f"\n{category}:")
    for step in steps:
        print(f"  {step}")

print("\n📚 Additional Resources:")
resources = [
    "• Paper: https://inria.hal.science/hal-04683475/file/eccv_2024_pose_independent_anthropometry_camera_ready.pdf",
    "• SMPL-Fitting: https://github.com/DavidBoja/SMPL-Fitting",
    "• SMPL Models: https://smpl.is.tue.mpg.de/",
    "• DYNA Dataset: http://dyna.is.tue.mpg.de/",
    "• 4DHumanOutfit: https://kinovis.inria.fr/4dhumanoutfit/"
]

for resource in resources:
    print(resource)

## 📝 Summary

This notebook provided a comprehensive introduction to the **Pose-Independent 3D Anthropometry** repository. Here's what we covered:

### ✅ What You Learned:
- **Project Goal**: Estimate 11 body measurements from 70 landmarks in any pose
- **Model Architecture**: SimpleMLP with coordinate and distance features
- **Datasets**: CAESAR, DYNA, and 4DHumanOutfit support
- **Training Process**: End-to-end pipeline from data to evaluation
- **Evaluation**: Multiple scenarios testing robustness

### 🎯 Key Features:
- **Pose Independence**: Works with arbitrary poses, not just A-pose
- **Sparse Input**: Only requires 70 anatomical landmarks
- **Robust**: Handles noise and different body types
- **Comprehensive**: Supports multiple datasets and evaluation scenarios

### 🔄 Typical Workflow:
1. **Setup**: Install dependencies and download required models
2. **Data**: Prepare datasets using provided scripts
3. **Train**: Use `train.py` with configuration in `config_real.yaml`
4. **Evaluate**: Test on various scenarios using `evaluate.py`
5. **Analyze**: Compare results with baselines and paper metrics

### 🚀 Ready to Start?
You're now equipped to:
- Use the pre-trained model for inference
- Train your own models with custom data
- Evaluate on different pose conditions
- Extend the method for your specific needs

**Happy researching! 🎉**